In [0]:
import mlflow
import mlflow.spark
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [0]:
temp_path = "/Volumes/workspace/ecommerce/silver_volume/mlflow_tmp"
dbutils.fs.mkdirs(temp_path)

print("UC temp directory ready:", temp_path)

In [0]:
train_df = spark.read.table("workspace.ecommerce.train_dataset")
test_df  = spark.read.table("workspace.ecommerce.test_dataset")

print("Train:", train_df.count())
print("Test :", test_df.count())

In [0]:
label_counts = train_df.groupBy("label").count().collect()

count_0 = [r['count'] for r in label_counts if r['label'] == 0][0]
count_1 = [r['count'] for r in label_counts if r['label'] == 1][0]

weight_0 = (count_0 + count_1) / (2 * count_0)
weight_1 = (count_0 + count_1) / (2 * count_1)

train_df = train_df.withColumn(
    "classWeight",
    F.when(F.col("label") == 1, weight_1).otherwise(weight_0)
)

In [0]:
feature_cols = [
    "total_events",
    "unique_products",
    "avg_price"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

train_ml = assembler.transform(train_df).select("features", "label", "classWeight")
test_ml  = assembler.transform(test_df).select("features", "label")

In [0]:
evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

In [0]:
print("===== Logistic Regression Runs =====")

for reg in [0.0, 0.01, 0.1]:

    with mlflow.start_run(run_name=f"LR_reg_{reg}"):

        lr = LogisticRegression(
            featuresCol="features",
            labelCol="label",
            weightCol="classWeight",
            regParam=reg,
            maxIter=20
        )

        model = lr.fit(train_ml)
        pred = model.transform(test_ml)
        auc = evaluator.evaluate(pred)

        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_param("regParam", reg)
        mlflow.log_metric("AUC", auc)

        mlflow.spark.log_model(
            model,
            artifact_path="model",
            dfs_tmpdir=temp_path
        )

        print(f"LR | reg={reg} | AUC={auc}")

In [0]:
print("===== RandomForest Runs =====")

for trees in [20, 50]:
    for depth in [5, 10]:

        with mlflow.start_run(run_name=f"RF_{trees}_trees_depth_{depth}"):

            rf = RandomForestClassifier(
                featuresCol="features",
                labelCol="label",
                numTrees=trees,
                maxDepth=depth,
                seed=42
            )

            model = rf.fit(train_ml)
            pred = model.transform(test_ml)
            auc = evaluator.evaluate(pred)

            mlflow.log_param("model_type", "RandomForest")
            mlflow.log_param("numTrees", trees)
            mlflow.log_param("maxDepth", depth)
            mlflow.log_metric("AUC", auc)

            mlflow.spark.log_model(
                model,
                artifact_path="model",
                dfs_tmpdir=temp_path
            )

            print(f"RF | trees={trees} depth={depth} | AUC={auc}")

In [0]:
import mlflow

mlflow.search_experiments()